## 1. Install and import libraries

In [1]:
%pip install -q transformers accelerate

import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Please enable GPU in Kaggle Notebook Settings.")

Note: you may need to restart the kernel to use updated packages.
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## 2. Experiment configuration

In [2]:
MODEL_NAME = "j-hartmann/emotion-english-roberta-large"

LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

EPOCHS = 3
LEARNING_RATE = 1e-5
MAX_LENGTH = 128
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
GRAD_ACCUMULATION = 8
WEIGHT_DECAY = 0.01
SEED = 42

DEVICE = torch.device("cuda")

print("Model:", MODEL_NAME)
print("Number of classes:", len(LABELS))
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUMULATION)

Model: j-hartmann/emotion-english-roberta-large
Number of classes: 6
Effective batch size: 16
Output directory: /kaggle/working/roberta_large


## 3. Load and prepare competition data

In [3]:
DATA_DIR = "/kaggle/input/competitions/2026-dm-lab-2-private-competition"

posts = pd.read_json(os.path.join(DATA_DIR, "final_posts.json"))
identification = pd.read_csv(os.path.join(DATA_DIR, "data_identification.csv"))
emotions = pd.read_csv(os.path.join(DATA_DIR, "emotion.csv"))
sample_submission = pd.read_csv(os.path.join(DATA_DIR, "sampleSubmission.csv"))

data = posts.merge(identification, on="post_id", how="inner", validate="one_to_one")
train_df = data[data["set"] == "train"].merge(
    emotions, on="post_id", how="inner", validate="one_to_one"
).reset_index(drop=True)

test_df = sample_submission[["post_id"]].merge(
    data[data["set"] == "test"][["post_id", "text"]],
    on="post_id",
    how="left",
    validate="one_to_one"
)

CURLY = str.maketrans({
    "\u2019": "'",
    "\u2018": "'",
    "\u201c": '"',
    "\u201d": '"'
})

def prep_for_roberta(text):
    text = "" if text is None or pd.isna(text) else str(text)
    text = text.translate(CURLY)
    text = text.replace("@<username>", "@user")
    text = text.replace("##URL##", "http")
    return " ".join(text.split())

train_df["text"] = train_df["text"].map(prep_for_roberta)
test_df["text"] = test_df["text"].map(prep_for_roberta)

X_train = train_df["text"].values
y_train = train_df["label"].map(LABEL2ID).values

print("Training samples:", len(X_train))
print("Test samples:", len(test_df))
print("\nClass distribution:")
print(train_df["label"].value_counts())
print("\nMissing text values:")
print(train_df["text"].isna().sum())
print("\nEmpty training texts:")
print((train_df["text"].str.len() == 0).sum())
print("\nEmpty test texts:")
print((test_df["text"].str.len() == 0).sum())
print("\nDuplicate training texts:", train_df["text"].duplicated().sum())


print("\nData preparation completed.")

Training samples: 3360
Test samples: 1440

Class distribution:
label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288
Name: count, dtype: int64

Missing text values:
0

Empty training texts:
0

Empty test texts:
0

Duplicate training texts: 50

Data preparation completed.


## 4. Dataset and helper functions

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class EmotionDataset(Dataset):
    def __init__(self, texts, tokenizer, labels=None):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH)
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(value[idx], dtype=torch.long)
            for key, value in self.encodings.items()
        }

        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)

        return item


def load_pretrained_model():
    config = AutoConfig.from_pretrained(MODEL_NAME)
    original_id2label = {int(i): str(label).lower() for i, label in config.id2label.items()}
    original_label2id = {label: idx for idx, label in original_id2label.items()}

    print("Original model labels:", original_id2label)
    missing_labels = set(LABELS) - set(original_label2id)
    if missing_labels:
        raise ValueError(f"Missing emotion labels: {missing_labels}")

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    classifier = model.classifier
    if not hasattr(classifier, "out_proj"):
        raise TypeError("Unexpected classifier architecture: expected classifier.out_proj")

    old_head = classifier.out_proj
    selected_rows = [original_label2id[label] for label in LABELS]
    new_head = nn.Linear(old_head.in_features, len(LABELS))

    with torch.no_grad():
        new_head.weight.copy_(old_head.weight[selected_rows])
        new_head.bias.copy_(old_head.bias[selected_rows])

    classifier.out_proj = new_head
    model.config.num_labels = len(LABELS)
    model.config.id2label = ID2LABEL
    model.config.label2id = LABEL2ID
    return model


@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    predictions = []

    for batch in loader:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(**batch).logits

        predictions.append(logits.float().cpu().numpy())

    return np.concatenate(predictions, axis=0)


print("Helper functions defined.")

Helper functions defined.


## 5. Verify pretrained classification head

In [5]:
set_seed(SEED)
test_model = load_pretrained_model()

print("\nModel:", MODEL_NAME)
print("Output labels:", test_model.config.id2label)
print("Number of output classes:", test_model.config.num_labels)
print("Classifier output layer:", test_model.classifier.out_proj)


print("\nPretrained emotion head successfully initialized.")

del test_model
gc.collect()
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


Model: j-hartmann/emotion-english-roberta-large
Output labels: {0: 'anger', 1: 'disgust', 2: 'fear', 3: 'joy', 4: 'sadness', 5: 'surprise'}
Number of output classes: 6
Classifier output layer: Linear(in_features=1024, out_features=6, bias=True)

Pretrained emotion head successfully initialized.


## 6. Train the final model

In [6]:
import math

set_seed(SEED)

X_full = train_df["text"].fillna("").astype(str).values
y_full = train_df["label"].map(LABEL2ID).values

print("Training samples:", len(X_full))
print("Label distribution:", np.bincount(y_full))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_dataset = EmotionDataset(X_full, tokenizer, labels=y_full)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator, pin_memory=True)

final_model = load_pretrained_model()
final_model.to(DEVICE)

class_counts = np.bincount(y_full, minlength=len(LABELS))
class_weights = (len(y_full) / (len(LABELS) * class_counts)) ** 0.5
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print("Class counts:", class_counts)
print("Class weights:", class_weights.cpu().numpy())

optimizer = torch.optim.AdamW(
    final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUMULATION)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

for epoch in range(EPOCHS):
    final_model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        group_start = (step // GRAD_ACCUMULATION) * GRAD_ACCUMULATION
        group_size = min(GRAD_ACCUMULATION, len(train_loader) - group_start)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            logits = final_model(
                input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]
            ).logits
            loss = criterion(logits, batch["labels"])
            scaled_loss = loss / group_size

        scaler.scale(scaled_loss).backward()
        total_loss += loss.item()
        is_update_step = (step + 1) % GRAD_ACCUMULATION == 0 or (step + 1) == len(train_loader)

        if is_update_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(final_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{EPOCHS} | Loss: {avg_loss:.4f}")

print("\nFinal training completed!")

Training samples: 3360
Label distribution: [674 454 288 875 628 441]
Original model labels: {0: 'anger', 1: 'disgust', 2: 'fear', 3: 'joy', 4: 'neutral', 5: 'sadness', 6: 'surprise'}


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Class counts: [674 454 288 875 628 441]
Class weights: [0.91151553 1.1106216  1.3944334  0.8        0.9443091  1.1268723 ]


/tmp/ipykernel_23/3933305252.py:185: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1/3 | Loss: 1.1176
Epoch 2/3 | Loss: 0.8918
Epoch 3/3 | Loss: 0.7559

Final training completed!


## 7. Generate submission

In [7]:
X_test = test_df["text"].fillna("").astype(str).values
test_dataset = EmotionDataset(X_test, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collator, pin_memory=True)

print("Test samples:", len(X_test))

final_test_scores = predict_logits(final_model, test_loader)
final_test_predictions = final_test_scores.argmax(axis=1)
predicted_labels = [ID2LABEL[int(pred)] for pred in final_test_predictions]

submission = pd.DataFrame({
    "id": sample_submission["post_id"].values,
    "label": predicted_labels
})
OUTPUT_PATH = "/kaggle/working/submission.csv"
submission.to_csv(OUTPUT_PATH, index=False)

print("\nSubmission generated!")
print("Shape:", submission.shape)
print("\nPredicted label distribution:")
print(submission["label"].value_counts())
print("\nPreview:")
display(submission.head())
print("\nSaved to:", OUTPUT_PATH)


print("\nAll submission checks passed!")

Test samples: 1440

Submission generated!
Shape: (1440, 2)

Predicted label distribution:
label
joy         382
anger       340
sadness     251
surprise    196
disgust     146
fear        125
Name: count, dtype: int64

Preview:


,id,label
0,xyzx9jp,surprise
1,l3vzzxt,sadness
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise



Saved to: /kaggle/working/submission.csv

All submission checks passed!
